# Model A: Mean-pooled embedding + MLP (baseline)

Baseline (textbook Ch 3-4: fully connected networks). Words are embedded, averaged over the sentence (order ignored), and passed through an MLP.

**Setup:** Runtime > Change runtime type > GPU (T4). Run all cells top to bottom. Results are appended to `results/results_A.csv`; do NOT edit `common.py` (frozen shared code).

In [ ]:
%pip install -q datasets transformers
import os
if not os.path.exists("common.py"):
    REPO_URL = "https://github.com/<ORG>/<REPO>.git"   # TODO: set to the group repo
    !git clone $REPO_URL project
    %cd project

In [ ]:
# Optional (Colab): keep results on Drive so they survive session timeouts.
# Must run BEFORE `import common`.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ["RESULTS_DIR"] = "/content/drive/MyDrive/CSCI167_results"
except Exception:
    pass  # running locally: results go to ./results

In [ ]:
OWNER = "A"
RESULTS_FILE = f"results_{OWNER}.csv"
SEEDS = [0, 1, 2]

import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from common import *

print(save_env_info(OWNER))                  # versions + hardware for the report
data = load_data()
print({k: len(v[0]) for k, v in data.items()}, "-> train/val/test sizes")
print("label counts (train):", np.bincount(data["train"][1]))

## Model

In [ ]:
class MeanPoolMLP(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden=128, dropout=0.5, n_hidden=1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        layers, d = [], emb_dim
        for _ in range(n_hidden):
            layers += [nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(dropout)]
            d = hidden
        layers.append(nn.Linear(d, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, ids, mask):
        m = mask.unsqueeze(-1).float()
        pooled = (self.emb(ids) * m).sum(1) / m.sum(1).clamp(min=1)
        return self.net(pooled)

set_seed(0)
_, _, _, v = make_vocab_loaders(data, 64, 0)
m = MeanPoolMLP(len(v)); print(m); print("trainable / total params:", count_params(m))

## Main runs (full training set, 3 seeds) and reduced-data runs (10%, 25%)

Shared protocol: Adam, lr 1e-3, batch 64, max 20 epochs, early stopping patience 5 on validation accuracy.

In [ ]:
def run(seed, frac):
    set_seed(seed)
    train_dl, val_dl, test_dl, vocab = make_vocab_loaders(data, 64, seed, train_frac=frac)
    model = MeanPoolMLP(len(vocab))
    return train_and_eval(model, (train_dl, val_dl, test_dl), run_name=f"mlp_f{frac}_s{seed}",
                          model_name="MeanPoolMLP", seed=seed, optimizer="adam", lr=1e-3,
                          epochs=20, patience=5, batch_size=64, train_frac=frac,
                          results_file=RESULTS_FILE)

for frac in [1.0, 0.25, 0.1]:
    for seed in SEEDS:
        run(seed, frac)

## Results: summary table, learning curves, error analysis

In [ ]:
summarize(f'results_{OWNER}.csv')   # mean +/- std over seeds, per configuration

In [ ]:
plot_history("mlp_f1.0_s0", "MeanPoolMLP: train vs validation (seed 0)");

In [ ]:
import json
h = json.load(open(f"{RESULTS_DIR}/history/mlp_f1.0_s0.json"))
preds, labels = np.array(h["test_preds"]), np.array(data["test"][1])
texts = data["test"][0]
cm = np.zeros((2, 2), int)
for y, p in zip(labels, preds): cm[y, p] += 1
print("confusion matrix (rows=true neg/pos, cols=pred neg/pos):\n", cm)
wrong = np.where(preds != labels)[0]
print(len(wrong), "errors. Examples (true, pred, text):")
for i in wrong[:20]: print(labels[i], preds[i], "|", texts[i])
# TODO for the report: what do the errors have in common? (sarcasm, negation, mixed reviews, very short text...)